In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
import warnings
import os

# Suppress Python warnings
warnings.filterwarnings('ignore')

# Suppress TensorFlow, standard error, and system-level logs
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
RADAR_ROOT = "/kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Radar"

# Activity labels
# ---------------------------------------------------------
ACTIVITIES = {
    0: "Wash_face",
    1: "Brush_teeth",
    2: "Comb_hair",
    3: "Take_off_clothes",
    4: "Wipe_hands",
    5: "Put_on_clothes",
    6: "Drink_water",
    7: "Eat_food",
    8: "Take_and_use_tableware",
    9: "Pour_drinks",
    10: "Stir_drinks",
    11: "Peel_fruits",
    12: "Sweep_the_floor",
    13: "Mop_the_floor",
    14: "Wipe_bowls",
    15: "Wipe_windows_and_tables",
    16: "Fold_clothes",
    17: "Tap_the_keyboard",
    18: "Write",
    19: "Make_a_phone_call",
    20: "Check_the_time",
    21: "Read_documents",
    22: "Turn_pages",
    23: "Listen_to_music_with_headphones",
    24: "Use_a_mobile_phone",
    25: "Watch_TV",
    26: "Play_games",
    27: "Take_a_selfie",
    28: "Jog_in_place",
    29: "Do_squats",
    30: "Do_jumping_jacks",
    31: "Do_stretching_exercises",
    32: "Stand_up",
    33: "Lie_down",
    34: "Sit_down",
    35: "Do_lunges",
    36: "Walk",
    37: "Take_medicine",
    38: "Massage_oneself",
    39: "Take_body_temperature",
}

files = glob.glob(os.path.join(RADAR_ROOT, "**", "*.csv"), recursive=True)

print(f"Total Radar CSV files: {len(files):,}")

# Scan
# ---------------------------------------------------------
records = []
value_stats = defaultdict(list)

for path in files:
    try:
        df = pd.read_csv(path)

        # Extract metadata from path
        parts = path.split(os.sep)
        activity_folder = next(
            (p for p in parts if p.split("_", 1)[0].isdigit()),
            None
        )
        user_folder = next(
            (p for p in parts if p.startswith("user")),
            None
        )

        if activity_folder is not None:
            class_id = int(activity_folder.split("_", 1)[0])
            activity = ACTIVITIES.get(class_id, activity_folder)
        else:
            class_id = -1
            activity = "UNKNOWN"

        user = user_folder if user_folder else "UNKNOWN"

        n_rows = len(df)

        if n_rows > 0 and "frame" in df.columns:
            n_frames = df["frame"].nunique()
            frames = df["frame"].to_numpy()

            frame_counts = df.groupby("frame").size()

            frame_min = int(frames.min())
            frame_max = int(frames.max())

            duration_frames = frame_max - frame_min + 1
            mean_det_frame = frame_counts.mean()
            max_det_frame = frame_counts.max()
            min_det_frame = frame_counts.min()
        else:
            n_frames = 0
            frame_min = np.nan
            frame_max = np.nan
            duration_frames = 0
            mean_det_frame = np.nan
            max_det_frame = np.nan
            min_det_frame = np.nan

        rec = {
            "path": path,
            "user": user,
            "class_id": class_id,
            "activity": activity,
            "rows": n_rows,
            "frames": n_frames,
            "duration_frames": duration_frames,
            "mean_det_frame": mean_det_frame,
            "min_det_frame": min_det_frame,
            "max_det_frame": max_det_frame,
        }

        for col in ["x", "y", "z", "v", "snr", "noise"]:
            if col in df.columns and len(df) > 0:
                vals = pd.to_numeric(df[col], errors="coerce").dropna()

                if len(vals):
                    rec[f"{col}_mean"] = vals.mean()
                    rec[f"{col}_std"] = vals.std()
                    rec[f"{col}_min"] = vals.min()
                    rec[f"{col}_max"] = vals.max()
                    rec[f"{col}_median"] = vals.median()

        records.append(rec)

    except Exception as e:
        print("ERROR:", path, e)

stats = pd.DataFrame(records)

# Basic overview
# ---------------------------------------------------------
print("\n" + "=" * 70)
print("BASIC OVERVIEW")
print("=" * 70)

print("Files:", len(stats))
print("Empty:", (stats["rows"] == 0).sum())
print("Non-empty:", (stats["rows"] > 0).sum())

print("\nNon-empty recording statistics:")
print(
    stats.loc[stats.rows > 0,
              ["rows", "frames", "duration_frames", "mean_det_frame"]]
    .describe()
    .round(2)
)

In [ ]:
# Feature distributions
# ---------------------------------------------------------
print("\n" + "=" * 70)
print("RADAR VALUE DISTRIBUTIONS")
print("=" * 70)

for col in ["x", "y", "z", "v", "snr", "noise"]:
    cols = [f"{col}_{s}" for s in ["mean", "std", "min", "max", "median"]]

    available = [c for c in cols if c in stats.columns]

    if available:
        print(f"\n--- {col.upper()} ---")
        print(stats.loc[stats.rows > 0, available].describe().round(3))

# ---------------------------------------------------------
# User statistics
# ---------------------------------------------------------
print("\n" + "=" * 70)
print("RADAR COVERAGE BY USER")
print("=" * 70)

user_stats = (
    stats.groupby("user")
    .agg(
        total=("rows", "size"),
        usable=("rows", lambda x: (x > 0).sum()),
        avg_rows=("rows", lambda x: x[x > 0].mean() if (x > 0).any() else 0),
        avg_frames=("frames", lambda x: x[x > 0].mean() if (x > 0).any() else 0),
    )
)

user_stats["availability_%"] = (
    100 * user_stats["usable"] / user_stats["total"]
)

print(user_stats.round(2).to_string())

# ---------------------------------------------------------
# Class statistics
# ---------------------------------------------------------
print("\n" + "=" * 70)
print("RADAR COVERAGE BY ACTIVITY")
print("=" * 70)

class_stats = (
    stats.groupby(["class_id", "activity"])
    .agg(
        total=("rows", "size"),
        usable=("rows", lambda x: (x > 0).sum()),
        avg_rows=("rows", lambda x: x[x > 0].mean() if (x > 0).any() else 0),
        avg_frames=("frames", lambda x: x[x > 0].mean() if (x > 0).any() else 0),
    )
)

class_stats["availability_%"] = (
    100 * class_stats["usable"] / class_stats["total"]
)

print(class_stats.round(2).to_string())

# ---------------------------------------------------------
# Longest / shortest sequences
# ---------------------------------------------------------
print("\n" + "=" * 70)
print("SEQUENCE LENGTH EXTREMES")
print("=" * 70)

usable_stats = stats[stats.rows > 0]

print("\nShortest:")
print(
    usable_stats
    .sort_values("rows")
    [["user", "class_id", "activity", "rows", "frames", "duration_frames"]]
    .head(15)
    .to_string(index=False)
)

print("\nLongest:")
print(
    usable_stats
    .sort_values("rows", ascending=False)
    [["user", "class_id", "activity", "rows", "frames", "duration_frames"]]
    .head(15)
    .to_string(index=False)
)

# ---------------------------------------------------------
# Save profiling table
# ---------------------------------------------------------
stats.to_csv("/kaggle/working/radar_recording_stats.csv", index=False)

print("\nSaved:")
print("/kaggle/working/radar_recording_stats.csv")

Total Radar CSV files: 2,914

BASIC OVERVIEW
Files: 2914
Empty: 1505
Non-empty: 1409

Non-empty recording statistics:
          rows   frames  duration_frames  mean_det_frame
count  1409.00  1409.00          1409.00         1409.00
mean    387.03    57.17            57.17            6.73
std     367.42    48.83            48.83            1.95
min      15.00     3.00             3.00            1.44
25%     155.00    24.00            24.00            5.46
50%     277.00    44.00            44.00            6.66
75%     507.00    75.00            75.00            7.93
max    4252.00   538.00           538.00           13.40

RADAR VALUE DISTRIBUTIONS

--- X ---
         x_mean     x_std     x_min     x_max  x_median
count  1409.000  1409.000  1409.000  1409.000  1409.000
mean     -0.166     0.762    -2.153     1.886    -0.041
std       0.227     0.358     1.071     0.897     0.087
min      -1.008     0.084    -5.409    -0.002    -1.061
25%      -0.254     0.471    -3.005     1.233    -0

# RADAR DEEP LEARNING V1
## PointNet-style Frame Encoder + Temporal TCN + Attention

In [ ]:
import os
import glob
import random
import copy
import math
import numpy as np
import pandas as pd

from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score


# ============================================================
# 1. CONFIG
# ============================================================

RADAR_ROOT = (
    "/kaggle/input/datasets/samasiayushman/"
    "small-model-track/Training/Training/data/Radar"
)

SEED = 42

NUM_CLASSES = 40

# Maximum number of frames retained from one recording
MAX_FRAMES = 128

# Maximum detections retained per frame
MAX_DETECTIONS = 32

BATCH_SIZE = 16

EPOCHS = 60

LEARNING_RATE = 3e-4

WEIGHT_DECAY = 1e-4

PATIENCE = 10

NUM_WORKERS = 2

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

# 2. REPRODUCIBILITY


In [ ]:
# ============================================================

def seed_everything(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


seed_everything(SEED)

# 3. DISCOVER RECORDINGS


In [ ]:
files = glob.glob(
    os.path.join(
        RADAR_ROOT,
        "**",
        "*.csv"
    ),
    recursive=True
)

records = []

for path in files:

    try:

        # ----------------------------------------------------
        # metadata
        # ----------------------------------------------------

        parts = path.split(os.sep)

        activity_folder = next(
            (
                p for p in parts
                if p.split("_", 1)[0].isdigit()
            ),
            None
        )

        user_folder = next(
            (
                p for p in parts
                if p.startswith("user")
            ),
            None
        )

        if activity_folder is None:
            continue

        class_id = int(
            activity_folder.split("_", 1)[0]
        )

        user = user_folder

        # ----------------------------------------------------
        # only keep non-empty files
        # ----------------------------------------------------

        df = pd.read_csv(path)

        if len(df) == 0:
            continue

        required = [
            "frame",
            "x",
            "y",
            "z",
            "v",
            "snr",
            "noise"
        ]

        if not all(c in df.columns for c in required):
            continue

        records.append({
            "path": path,
            "class_id": class_id,
            "user": user
        })

    except Exception:
        pass


records_df = pd.DataFrame(records)

print()
print("=" * 70)
print("DATASET")
print("=" * 70)

print("Usable recordings:", len(records_df))
print("Users:", sorted(records_df.user.unique()))

print("\nClass distribution:")
print(
    records_df["class_id"]
    .value_counts()
    .sort_index()
)

# 4. USER-INDEPENDENT TRAIN / VALIDATION SPLIT


In [ ]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

train_idx, val_idx = next(
    gss.split(
        records_df,
        records_df["class_id"],
        groups=records_df["user"]
    )
)

train_df = records_df.iloc[train_idx].reset_index(drop=True)

val_df = records_df.iloc[val_idx].reset_index(drop=True)


print()
print("=" * 70)
print("USER-INDEPENDENT SPLIT")
print("=" * 70)

print("Train:", len(train_df))
print("Val  :", len(val_df))

print("\nTrain users:")
print(sorted(train_df.user.unique()))

print("\nVal users:")
print(sorted(val_df.user.unique()))

print(
    "\nOverlap:",
    set(train_df.user) & set(val_df.user)
)

# 5. RADAR NORMALIZATION


In [ ]:
# Physical Radar features have very different scales:
#
# x,y,z ~ units
# v     ~ fractions
# snr   ~ hundreds
# noise ~ hundreds
#
# We calculate normalization statistics from TRAIN ONLY.
# ============================================================

RAW_COLS = [
    "x",
    "y",
    "z",
    "v",
    "snr",
    "noise"
]

all_values = []

print()
print("Calculating training normalization statistics...")

for path in train_df["path"]:

    df = pd.read_csv(path)

    vals = (
        df[RAW_COLS]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .values
    )

    if len(vals):
        all_values.append(vals)


all_values = np.concatenate(
    all_values,
    axis=0
)

feature_mean = np.mean(
    all_values,
    axis=0
)

feature_std = np.std(
    all_values,
    axis=0
)

feature_std = np.maximum(
    feature_std,
    1e-6
)

print("\nMean:")
print(
    dict(
        zip(
            RAW_COLS,
            feature_mean.round(4)
        )
    )
)

print("\nStd:")
print(
    dict(
        zip(
            RAW_COLS,
            feature_std.round(4)
        )
    )
)

# 6. DATASET


In [ ]:
class RadarDataset(Dataset):

    def __init__(
        self,
        dataframe,
        mean,
        std,
        max_frames=128,
        max_detections=32,
        augment=False
    ):

        self.df = dataframe.reset_index(drop=True)

        self.mean = np.asarray(
            mean,
            dtype=np.float32
        )

        self.std = np.asarray(
            std,
            dtype=np.float32
        )

        self.max_frames = max_frames
        self.max_detections = max_detections

        self.augment = augment


    def __len__(self):

        return len(self.df)


    def load_sequence(self, path):

        df = pd.read_csv(path)

        df = df[
            [
                "frame",
                "x",
                "y",
                "z",
                "v",
                "snr",
                "noise"
            ]
        ].copy()

        df["frame"] = pd.to_numeric(
            df["frame"],
            errors="coerce"
        )

        for c in RAW_COLS:

            df[c] = pd.to_numeric(
                df[c],
                errors="coerce"
            )

        df = df.dropna()

        if len(df) == 0:

            return np.zeros(
                (
                    self.max_frames,
                    self.max_detections,
                    6
                ),
                dtype=np.float32
            ), np.zeros(
                self.max_frames,
                dtype=np.bool_
            )


        # ----------------------------------------------------
        # group detections by frame
        # ----------------------------------------------------

        frame_ids = sorted(
            df["frame"].unique()
        )


        # ----------------------------------------------------
        # temporal sampling
        # ----------------------------------------------------

        if len(frame_ids) > self.max_frames:

            # Training:
            # randomly sample a contiguous temporal window.
            #
            # Validation:
            # deterministic center window.

            if self.augment:

                start = np.random.randint(
                    0,
                    len(frame_ids) - self.max_frames + 1
                )

            else:

                start = (
                    len(frame_ids) -
                    self.max_frames
                ) // 2

            frame_ids = frame_ids[
                start:
                start + self.max_frames
            ]


        # ----------------------------------------------------
        # tensors
        # ----------------------------------------------------

        sequence = np.zeros(
            (
                self.max_frames,
                self.max_detections,
                6
            ),
            dtype=np.float32
        )

        frame_mask = np.zeros(
            self.max_frames,
            dtype=np.bool_
        )


        # ----------------------------------------------------
        # construct frame tensors
        # ----------------------------------------------------

        for t, frame_id in enumerate(frame_ids):

            if t >= self.max_frames:
                break

            frame_df = df[
                df["frame"] == frame_id
            ]

            values = frame_df[
                RAW_COLS
            ].values.astype(
                np.float32
            )


            # ------------------------------------------------
            # If more detections than allowed:
            # choose strongest detections by SNR.
            # ------------------------------------------------

            if len(values) > self.max_detections:

                snr_idx = 4

                order = np.argsort(
                    values[:, snr_idx]
                )[::-1]

                values = values[
                    order[:self.max_detections]
                ]


            # ------------------------------------------------
            # normalize
            # ------------------------------------------------

            values = (
                values - self.mean
            ) / self.std


            n = min(
                len(values),
                self.max_detections
            )

            sequence[
                t,
                :n
            ] = values[:n]

            frame_mask[t] = True


        # ----------------------------------------------------
        # simple Radar augmentation
        # ----------------------------------------------------

        if self.augment:

            # Small Gaussian measurement noise
            if np.random.rand() < 0.5:

                noise = np.random.normal(
                    0,
                    0.015,
                    size=sequence.shape
                ).astype(np.float32)

                sequence += noise * (
                    frame_mask[:, None, None]
                )


            # Detection dropout
            if np.random.rand() < 0.30:

                for t in range(
                    self.max_frames
                ):

                    if not frame_mask[t]:
                        continue

                    keep = (
                        np.random.rand(
                            self.max_detections
                        ) > 0.08
                    )

                    sequence[t, ~keep] = 0


        return sequence, frame_mask


    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        sequence, frame_mask = (
            self.load_sequence(
                row["path"]
            )
        )

        label = int(
            row["class_id"]
        )

        return (
            torch.tensor(
                sequence,
                dtype=torch.float32
            ),
            torch.tensor(
                frame_mask,
                dtype=torch.bool
            ),
            torch.tensor(
                label,
                dtype=torch.long
            )
        )

# 7. DATA LOADERS


In [ ]:
train_dataset = RadarDataset(
    train_df,
    feature_mean,
    feature_std,
    MAX_FRAMES,
    MAX_DETECTIONS,
    augment=True
)

val_dataset = RadarDataset(
    val_df,
    feature_mean,
    feature_std,
    MAX_FRAMES,
    MAX_DETECTIONS,
    augment=False
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(
        NUM_WORKERS > 0
    )
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(
        NUM_WORKERS > 0
    )
)

# 8. POINT / DETECTION ENCODER


In [ ]:
class DetectionEncoder(nn.Module):

    """
    PointNet-style encoder.

    Each radar detection:
        [x,y,z,v,snr,noise]
                   ↓
                MLP
                   ↓
              embedding

    Then max + mean pooling over detections
    produces one representation per frame.
    """

    def __init__(
        self,
        in_dim=6,
        hidden=64,
        out_dim=128
    ):

        super().__init__()

        self.mlp = nn.Sequential(

            nn.Linear(
                in_dim,
                hidden
            ),

            nn.BatchNorm1d(
                hidden
            ),

            nn.GELU(),

            nn.Linear(
                hidden,
                hidden
            ),

            nn.BatchNorm1d(
                hidden
            ),

            nn.GELU(),

            nn.Linear(
                hidden,
                out_dim
            ),

            nn.GELU()
        )


    def forward(
        self,
        x,
        detection_mask
    ):

        # x:
        # [B,T,D,F]

        B, T, D, Fdim = x.shape

        x = x.reshape(
            B * T * D,
            Fdim
        )

        z = self.mlp(x)

        z = z.reshape(
            B,
            T,
            D,
            -1
        )

        mask = detection_mask.unsqueeze(
            -1
        )

        # ----------------------------------------------------
        # Mean pooling
        # ----------------------------------------------------

        z_masked = z * mask

        count = mask.sum(
            dim=2
        ).clamp(
            min=1
        )

        mean_pool = (
            z_masked.sum(
                dim=2
            ) / count
        )

        # ----------------------------------------------------
        # Max pooling
        # ----------------------------------------------------

        z_for_max = z.masked_fill(
            ~mask,
            -1e9
        )

        max_pool = z_for_max.max(
            dim=2
        ).values

        # ----------------------------------------------------
        # Combine
        # ----------------------------------------------------

        frame_embedding = torch.cat(
            [
                mean_pool,
                max_pool
            ],
            dim=-1
        )

        return frame_embedding

# 9. TCN BLOCK


In [ ]:
class TCNBlock(nn.Module):

    def __init__(
        self,
        channels,
        dilation,
        dropout=0.15
    ):

        super().__init__()

        padding = dilation

        self.conv1 = nn.Conv1d(
            channels,
            channels,
            kernel_size=3,
            padding=padding,
            dilation=dilation
        )

        self.bn1 = nn.BatchNorm1d(
            channels
        )

        self.conv2 = nn.Conv1d(
            channels,
            channels,
            kernel_size=3,
            padding=padding,
            dilation=dilation
        )

        self.bn2 = nn.BatchNorm1d(
            channels
        )

        self.dropout = nn.Dropout(
            dropout
        )


    def forward(self, x):

        residual = x

        x = self.conv1(x)

        x = self.bn1(x)

        x = F.gelu(x)

        x = self.dropout(x)

        x = self.conv2(x)

        x = self.bn2(x)

        x = F.gelu(x)

        x = self.dropout(x)

        # Conv padding produces same length.
        return x + residual


# ============================================================
# 10. ATTENTION POOLING
# ============================================================

class AttentionPooling(nn.Module):

    def __init__(
        self,
        dim
    ):

        super().__init__()

        self.score = nn.Sequential(

            nn.Linear(
                dim,
                dim // 2
            ),

            nn.Tanh(),

            nn.Linear(
                dim // 2,
                1
            )
        )


    def forward(
        self,
        x,
        mask
    ):

        # x:
        # [B,T,C]

        scores = self.score(
            x
        ).squeeze(-1)

        scores = scores.masked_fill(
            ~mask,
            -1e9
        )

        weights = torch.softmax(
            scores,
            dim=1
        )

        pooled = torch.sum(
            x * weights.unsqueeze(-1),
            dim=1
        )

        return pooled


# ============================================================
# 11. COMPLETE RADAR MODEL
# ============================================================

class RadarNet(nn.Module):

    def __init__(
        self,
        num_classes=40
    ):

        super().__init__()

        # ----------------------------------------------------
        # Point encoder
        # ----------------------------------------------------

        self.point_encoder = (
            DetectionEncoder(
                in_dim=6,
                hidden=64,
                out_dim=128
            )
        )

        # 128 mean + 128 max
        frame_dim = 256

        # ----------------------------------------------------
        # Project frame representation
        # ----------------------------------------------------

        self.frame_projection = nn.Sequential(

            nn.Linear(
                frame_dim,
                256
            ),

            nn.LayerNorm(
                256
            ),

            nn.GELU(),

            nn.Dropout(
                0.15
            )
        )

        # ----------------------------------------------------
        # Temporal network
        # ----------------------------------------------------

        self.tcn = nn.Sequential(

            TCNBlock(
                256,
                dilation=1
            ),

            TCNBlock(
                256,
                dilation=2
            ),

            TCNBlock(
                256,
                dilation=4
            ),

            TCNBlock(
                256,
                dilation=8
            )
        )

        # ----------------------------------------------------
        # Attention
        # ----------------------------------------------------

        self.attention = AttentionPooling(
            256
        )

        # ----------------------------------------------------
        # Classifier
        # ----------------------------------------------------

        self.classifier = nn.Sequential(

            nn.Linear(
                256,
                128
            ),

            nn.LayerNorm(
                128
            ),

            nn.GELU(),

            nn.Dropout(
                0.30
            ),

            nn.Linear(
                128,
                num_classes
            )
        )


    def forward(
        self,
        x,
        frame_mask
    ):

        # x:
        # [B,T,D,6]

        B, T, D, Fdim = x.shape

        # ----------------------------------------------------
        # detection mask
        # ----------------------------------------------------

        detection_mask = (
            x.abs().sum(
                dim=-1
            ) > 0
        )

        # ----------------------------------------------------
        # PointNet frame encoding
        # ----------------------------------------------------

        frames = self.point_encoder(
            x,
            detection_mask
        )

        # ----------------------------------------------------
        # Frame projection
        # ----------------------------------------------------

        frames = self.frame_projection(
            frames
        )

        # [B,T,C] → [B,C,T]
        frames = frames.transpose(
            1,
            2
        )

        # ----------------------------------------------------
        # TCN
        # ----------------------------------------------------

        frames = self.tcn(
            frames
        )

        # [B,C,T] → [B,T,C]
        frames = frames.transpose(
            1,
            2
        )

        # ----------------------------------------------------
        # Attention pooling
        # ----------------------------------------------------

        pooled = self.attention(
            frames,
            frame_mask
        )

        # ----------------------------------------------------
        # Classifier
        # ----------------------------------------------------

        logits = self.classifier(
            pooled
        )

        return logits

# 12. MODEL & Weights


In [ ]:
model = RadarNet(
    num_classes=NUM_CLASSES
).to(DEVICE)

print()
print("=" * 70)
print("MODEL")
print("=" * 70)

print(model)

n_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    f"\nTrainable parameters: {n_params:,}"
)



train_counts = np.bincount(
    train_df["class_id"].values,
    minlength=NUM_CLASSES
)

# Effective inverse-frequency weighting
weights = np.zeros(
    NUM_CLASSES,
    dtype=np.float32
)

nonzero = train_counts > 0

weights[nonzero] = (
    1.0 /
    np.sqrt(
        train_counts[nonzero]
    )
)

weights = weights / weights[nonzero].mean()

class_weights = torch.tensor(
    weights,
    dtype=torch.float32,
    device=DEVICE
)

print("\nClass weights:")
print(
    np.round(
        weights,
        3
    )
)


# ============================================================
# 14. LOSS
# ============================================================

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.05
)


# ============================================================
# 15. OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


# ============================================================
# 16. LR SCHEDULER
# ============================================================

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=1e-6
)


# ============================================================
# 17. MIXED PRECISION
# ============================================================

use_amp = DEVICE.type == "cuda"

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp
)

# 18. TRAINING FUNCTION


In [ ]:

def train_one_epoch(
    model,
    loader,
    optimizer
):

    model.train()

    total_loss = 0.0

    all_preds = []
    all_targets = []

    for x, frame_mask, yb in loader:

        x = x.to(
            DEVICE,
            non_blocking=True
        )

        frame_mask = frame_mask.to(
            DEVICE,
            non_blocking=True
        )

        yb = yb.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=use_amp
        ):

            logits = model(
                x,
                frame_mask
            )

            loss = criterion(
                logits,
                yb
            )

        scaler.scale(
            loss
        ).backward()

        scaler.unscale_(
            optimizer
        )

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        scaler.step(
            optimizer
        )

        scaler.update()

        total_loss += (
            loss.item() *
            len(yb)
        )

        preds = logits.argmax(
            dim=1
        )

        all_preds.extend(
            preds.detach()
            .cpu()
            .numpy()
        )

        all_targets.extend(
            yb.detach()
            .cpu()
            .numpy()
        )

    loss = (
        total_loss /
        len(loader.dataset)
    )

    acc = accuracy_score(
        all_targets,
        all_preds
    )

    f1 = f1_score(
        all_targets,
        all_preds,
        average="macro",
        zero_division=0
    )

    return loss, acc, f1

# 19. VALIDATION


In [ ]:


@torch.no_grad()
def validate(
    model,
    loader
):

    model.eval()

    total_loss = 0.0

    all_preds = []
    all_targets = []

    for x, frame_mask, yb in loader:

        x = x.to(
            DEVICE,
            non_blocking=True
        )

        frame_mask = frame_mask.to(
            DEVICE,
            non_blocking=True
        )

        yb = yb.to(
            DEVICE,
            non_blocking=True
        )

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=use_amp
        ):

            logits = model(
                x,
                frame_mask
            )

            loss = criterion(
                logits,
                yb
            )

        total_loss += (
            loss.item() *
            len(yb)
        )

        preds = logits.argmax(
            dim=1
        )

        all_preds.extend(
            preds.cpu().numpy()
        )

        all_targets.extend(
            yb.cpu().numpy()
        )

    loss = (
        total_loss /
        len(loader.dataset)
    )

    acc = accuracy_score(
        all_targets,
        all_preds
    )

    balanced_acc = balanced_accuracy_score(
        all_targets,
        all_preds
    )

    f1 = f1_score(
        all_targets,
        all_preds,
        average="macro",
        zero_division=0
    )

    return (
        loss,
        acc,
        balanced_acc,
        f1,
        np.array(all_targets),
        np.array(all_preds)
    )


# ============================================================
# 20. TRAIN
# ============================================================

best_acc = -1

best_state = None

epochs_without_improvement = 0

history = []

Device: cpu

DATASET
Usable recordings: 1409
Users: ['user1', 'user2', 'user3', 'user4', 'user5', 'user6', 'user7', 'user8', 'user9']

Class distribution:
class_id
0      15
1      13
2      21
3      17
4      29
5      21
6      72
7      92
8      65
9      69
10     64
11     67
12     29
13     28
14     20
15     21
16      6
17     42
18     19
19     22
20     40
21     34
22     28
23     31
24     20
25      6
26     14
27     15
28     20
29     32
30     23
31     43
32     41
33     18
34     83
35     12
36    150
37     30
38     18
39     19
Name: count, dtype: int64

USER-INDEPENDENT SPLIT
Train: 1090
Val  : 319

Train users:
['user1', 'user3', 'user4', 'user5', 'user6', 'user7', 'user9']

Val users:
['user2', 'user8']

Overlap: set()

Calculating training normalization statistics...

Mean:
{'x': np.float64(-0.1732), 'y': np.float64(1.0887), 'z': np.float64(-0.1687), 'v': np.float64(-0.0007), 'snr': np.float64(173.842), 'noise': np.float64(526.1173)}

Std:
{'x': np.flo

In [ ]:
print()
print("=" * 70)
print("TRAINING")
print("=" * 70)

for epoch in range(
    1,
    EPOCHS + 1
):

    train_loss, train_acc, train_f1 = (
        train_one_epoch(
            model,
            train_loader,
            optimizer
        )
    )

    (
        val_loss,
        val_acc,
        val_bal_acc,
        val_f1,
        val_targets,
        val_preds
    ) = validate(
        model,
        val_loader
    )

    scheduler.step()

    lr = optimizer.param_groups[0]["lr"]

    history.append({

        "epoch": epoch,

        "train_loss": train_loss,
        "train_acc": train_acc,
        "train_f1": train_f1,

        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_bal_acc": val_bal_acc,
        "val_f1": val_f1,

        "lr": lr
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss {train_loss:.4f} | "
        f"Train Acc {train_acc:.4f} | "
        f"Val Loss {val_loss:.4f} | "
        f"Val Acc {val_acc:.4f} | "
        f"Val BalAcc {val_bal_acc:.4f} | "
        f"Val F1 {val_f1:.4f} | "
        f"LR {lr:.2e}"
    )

    # --------------------------------------------------------
    # Save best validation accuracy
    # --------------------------------------------------------

    if val_acc > best_acc:

        best_acc = val_acc

        best_state = copy.deepcopy(
            model.state_dict()
        )

        torch.save(
            {
                "model_state_dict":
                    best_state,

                "feature_mean":
                    feature_mean,

                "feature_std":
                    feature_std,

                "max_frames":
                    MAX_FRAMES,

                "max_detections":
                    MAX_DETECTIONS,

                "val_acc":
                    val_acc,

                "val_f1":
                    val_f1,

                "epoch":
                    epoch
            },
            "/kaggle/working/radar_pointnet_tcn_best.pt"
        )

        epochs_without_improvement = 0

        print(
            f"  ✓ New best: "
            f"{val_acc:.4f}"
        )

    else:

        epochs_without_improvement += 1

    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------

    if epochs_without_improvement >= PATIENCE:

        print(
            f"\nEarly stopping at epoch "
            f"{epoch}"
        )

        break


# 21. RESTORE BEST MODEL
# ============================================================

if best_state is not None:

    model.load_state_dict(
        best_state
    )


# 22. FINAL VALIDATION
# ============================================================

(
    val_loss,
    val_acc,
    val_bal_acc,
    val_f1,
    val_targets,
    val_preds
) = validate(
    model,
    val_loader
)

print()
print("=" * 70)
print("BEST RADAR DL RESULT")
print("=" * 70)

print(
    f"Validation Accuracy : {val_acc:.4f}"
)

print(
    f"Balanced Accuracy   : {val_bal_acc:.4f}"
)

print(
    f"Macro F1            : {val_f1:.4f}"
)

print(
    f"Best checkpoint     : "
    f"/kaggle/working/radar_pointnet_tcn_best.pt"
)



# 23. CONFUSION MATRIX
# ============================================================

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    val_targets,
    val_preds,
    labels=np.arange(NUM_CLASSES)
)

print()
print("=" * 70)
print("PER-CLASS VALIDATION ACCURACY")
print("=" * 70)

for c in range(NUM_CLASSES):

    mask = val_targets == c

    if mask.sum() == 0:
        continue

    class_acc = (
        val_preds[mask] == c
    ).mean()

    print(
        f"{c:02d}: "
        f"{class_acc:.3f} "
        f"(n={mask.sum():3d})"
    )


# 24. SAVE HISTORY
# ============================================================

history_df = pd.DataFrame(
    history
)

history_df.to_csv(
    "/kaggle/working/radar_training_history_v1.csv",
    index=False
)

print(
    "\nSaved history:"
    "/kaggle/working/radar_training_history_v1.csv"
)


TRAINING
Epoch 01 | Train Loss 3.6417 | Train Acc 0.0917 | Val Loss 3.4694 | Val Acc 0.1097 | Val BalAcc 0.0701 | Val F1 0.0315 | LR 3.00e-04
  ✓ New best: 0.1097
Epoch 02 | Train Loss 3.4080 | Train Acc 0.1330 | Val Loss 3.4381 | Val Acc 0.1066 | Val BalAcc 0.0641 | Val F1 0.0381 | LR 2.99e-04
Epoch 03 | Train Loss 3.2312 | Train Acc 0.1752 | Val Loss 3.3879 | Val Acc 0.1129 | Val BalAcc 0.0814 | Val F1 0.0670 | LR 2.98e-04
  ✓ New best: 0.1129
Epoch 04 | Train Loss 3.1083 | Train Acc 0.1954 | Val Loss 3.2521 | Val Acc 0.1661 | Val BalAcc 0.1712 | Val F1 0.0973 | LR 2.97e-04
  ✓ New best: 0.1661
Epoch 05 | Train Loss 3.0226 | Train Acc 0.2037 | Val Loss 3.2442 | Val Acc 0.1599 | Val BalAcc 0.1234 | Val F1 0.0766 | LR 2.95e-04
Epoch 06 | Train Loss 2.9337 | Train Acc 0.2294 | Val Loss 3.2150 | Val Acc 0.1411 | Val BalAcc 0.1188 | Val F1 0.0805 | LR 2.93e-04
Epoch 07 | Train Loss 2.8411 | Train Acc 0.2578 | Val Loss 3.2668 | Val Acc 0.1536 | Val BalAcc 0.0951 | Val F1 0.0754 | LR 2.90e